# Text-to-SQL with the Transformer

**Generative AI - Assignment 02**

Encoder-decoder Transformer (Vaswani et al., 2017), implemented from scratch,
trained on WikiSQL to map an English question over a table to a SQL query.

**Allowed:** `torch`, `nn.Linear`, `nn.Embedding`, `nn.LayerNorm`,
`nn.Dropout`, `nn.ReLU`, `torch.softmax`, `torch.matmul`, the given starter
code.
**Forbidden (not used anywhere below):** `nn.Transformer`,
`nn.TransformerEncoder(Layer)`, `nn.TransformerDecoder(Layer)`,
`nn.MultiheadAttention`, `F.scaled_dot_product_attention`, Hugging Face
`transformers`, pretrained weights.

**To run:** Settings -> Accelerator: GPU, Internet: On -> Run All.


## Setup

In [ ]:
# Kaggle/Colab already ship torch, numpy and matplotlib.
# sentencepiece plus the WikiSQL evaluator's own tiny dependencies:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "sentencepiece", "records", "babel", "tqdm", "tabulate"], check=False)


In [ ]:
import json, math, random, time, subprocess
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import sentencepiece as spm

torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
print("torch", torch.__version__, "| device:", DEVICE, "| GPU:", gpu_name)


## Task 1 - Run the starter code

### 1.1 - Get the data

WikiSQL (Zhong, Xiong and Socher, 2017): 80,654 questions over 24,241 tables;
the official splits share no tables. Repository: https://github.com/salesforce/WikiSQL
Paper: https://arxiv.org/abs/1709.00103

The next cell downloads and extracts it automatically.

In [ ]:
!git clone -q https://github.com/salesforce/WikiSQL
!cd WikiSQL && tar xjf data.tar.bz2

CANDIDATE_DIRS = [
    Path("/kaggle/input/wikisql/data"),
    Path("/kaggle/input/wikisql/WikiSQL/data"),
    Path("WikiSQL/data"),
]
WIKISQL_DATA_DIR = next((d for d in CANDIDATE_DIRS if (d / "train.jsonl").exists()), None)
USE_REAL_DATA = WIKISQL_DATA_DIR is not None

if USE_REAL_DATA:
    print("Found real WikiSQL data at:", WIKISQL_DATA_DIR)
else:
    print("No real WikiSQL data found in", [str(d) for d in CANDIDATE_DIRS])
    print("Falling back to a tiny SYNTHETIC dataset (same schema) so the rest of this")
    print("notebook can still be smoke-tested. Get the real data for the graded run.")


### Fallback dataset

If real WikiSQL is not found, a tiny synthetic dataset with the same JSON
schema is generated instead, so the pipeline below still runs end to end.

In [ ]:
AGG_OPS_PREVIEW = ["", "MAX", "MIN", "COUNT", "SUM", "AVG"]  # (also defined properly below)

FIRST_NAMES = ["Terrence", "Kim", "James", "Ava", "Noah", "Liam", "Mia", "Omar", "Sara", "Wei",
               "Ivan", "Zoe", "Carlos", "Nina", "Yusuf", "Elena", "Hugo", "Priya", "Leo", "Mona"]
LAST_NAMES = ["Ross", "Manners", "Bond", "Stone", "Khan", "Silva", "Park", "Ahmed", "Lopez", "Chen"]
POSITIONS = ["Guard", "Forward", "Center", "Striker", "Defender", "Midfielder"]
TEAMS = ["Duke", "UCLA", "Kansas", "Kentucky", "Gonzaga", "Villanova", "Baylor", "Arizona"]
NATIONALITIES = ["USA", "Canada", "Nigeria", "Spain", "Turkey", "Australia", "France", "Brazil"]
HEADER_POOL = ["Player", "No.", "Nationality", "Position", "Years", "School/Club Team"]


def _make_table(table_id, n_rows=8):
    rows = [[f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}",
              str(random.randint(0, 55)), random.choice(NATIONALITIES), random.choice(POSITIONS),
              f"{random.randint(2000, 2015)}-{random.randint(2016, 2024)}", random.choice(TEAMS)]
             for _ in range(n_rows)]
    return {"id": table_id, "header": HEADER_POOL, "types": ["text"] * len(HEADER_POOL), "rows": rows}


def _make_example(table):
    row = random.choice(table["rows"])
    sel = random.randint(0, len(table["header"]) - 1)
    agg = random.choice([0, 2, 3]) if table["header"][sel] == "No." else 0
    cond_cols = random.sample(range(len(table["header"])), random.choice([1, 1, 2]))
    conds = [[c, 0, row[c]] for c in cond_cols]
    verb = f"the {AGG_OPS_PREVIEW[agg].lower()} {table['header'][sel]}" if agg else f"the {table['header'][sel]}"
    question = f"What is {verb} of the player where " + \
               " and ".join(f"{table['header'][c]} is {row[c]}" for c in cond_cols) + "?"
    return {"table_id": table["id"], "question": question, "sql": {"sel": sel, "agg": agg, "conds": conds}}


def _make_split(data_dir, prefix, n_tables, n_examples):
    tables = {f"{prefix}-{i}": _make_table(f"{prefix}-{i}") for i in range(n_tables)}
    examples = [_make_example(random.choice(list(tables.values()))) for _ in range(n_examples)]
    with open(data_dir / f"{prefix}.tables.jsonl", "w", encoding="utf-8") as f:
        for t in tables.values():
            f.write(json.dumps(t) + "\n")
    with open(data_dir / f"{prefix}.jsonl", "w", encoding="utf-8") as f:
        for e in examples:
            f.write(json.dumps(e) + "\n")


if not USE_REAL_DATA:
    WIKISQL_DATA_DIR = Path("synthetic_data/WikiSQL/data")
    WIKISQL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    _make_split(WIKISQL_DATA_DIR, "train", n_tables=40, n_examples=400)
    _make_split(WIKISQL_DATA_DIR, "dev", n_tables=10, n_examples=80)
    _make_split(WIKISQL_DATA_DIR, "test", n_tables=10, n_examples=80)
    print("Synthetic data written to", WIKISQL_DATA_DIR)

VOCAB_SIZE = 8000 if USE_REAL_DATA else 400  # 8000 per the assignment; smaller only for the tiny synthetic fallback


### 1.2 - `data_prep.py` (starter code, unchanged)

In [ ]:
DATA_DIR = WIKISQL_DATA_DIR

AGG_OPS = ["", "MAX", "MIN", "COUNT", "SUM", "AVG"]
COND_OPS = ["=", ">", "<"]
MAX_COLS = 64  # column tokens <c0> ... <c63>


def load_split(split):
    """Return (examples, tables) for 'train', 'dev' or 'test'."""
    tables = {}
    with open(DATA_DIR / f"{split}.tables.jsonl", encoding="utf-8") as f:
        for line in f:
            t = json.loads(line)
            tables[t["id"]] = t
    with open(DATA_DIR / f"{split}.jsonl", encoding="utf-8") as f:
        examples = [json.loads(line) for line in f]
    return examples, tables


def encode_source(question, header):
    """question <sep> <c0> col name <c1> col name ..."""
    cols = " ".join(f"<c{i}> {name}" for i, name in enumerate(header))
    return f"{question.strip()} <sep> {cols}".lower()


def encode_target(sql):
    """{'sel','agg','conds'} -> 'select count <c3> where <c1> = kim manners'"""
    out = ["select"]
    if sql["agg"]:
        out.append(AGG_OPS[sql["agg"]].lower())
    out.append(f"<c{sql['sel']}>")
    for i, (col, op, val) in enumerate(sql["conds"]):
        out += ["where" if i == 0 else "and", f"<c{col}>", COND_OPS[op], str(val)]
    return " ".join(out).lower()


def build_pairs(split):
    examples, tables = load_split(split)
    pairs = []
    for ex in examples:
        header = tables[ex["table_id"]]["header"]
        pairs.append({
            "table_id": ex["table_id"],
            "src": encode_source(ex["question"], header),
            "tgt": encode_target(ex["sql"]),
        })
    return pairs


PAIRS_DIR = Path("pairs")
PAIRS_DIR.mkdir(exist_ok=True)

all_pairs = {}
for split in ["train", "dev", "test"]:
    pairs = build_pairs(split)
    all_pairs[split] = pairs
    with open(PAIRS_DIR / f"{split}_pairs.jsonl", "w", encoding="utf-8") as f:
        for p in pairs:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(f"{split}: {len(pairs)} pairs")

print()
print(all_pairs["train"][0]["src"])
print(all_pairs["train"][0]["tgt"])


**Task 1.3 (part 1) - pairs per split, word-level source/target length**

In [ ]:
def _word_len_stats(pairs, key):
    lens = [len(p[key].split()) for p in pairs]
    return len(lens), np.mean(lens), np.max(lens)

print(f"{'split':6s} {'pairs':>6s} {'mean src':>9s} {'max src':>8s} {'mean tgt':>9s} {'max tgt':>8s}")
for split in ["train", "dev", "test"]:
    n, mean_s, max_s = _word_len_stats(all_pairs[split], "src")
    _, mean_t, max_t = _word_len_stats(all_pairs[split], "tgt")
    print(f"{split:6s} {n:6d} {mean_s:9.1f} {max_s:8.0f} {mean_t:9.1f} {max_t:8.0f}")


### 1.2 (cont.) - `tokenizer.py` (starter code, unchanged)

Source and target share one BPE vocabulary, exactly as in the paper, so the
encoder embedding, decoder embedding and output projection can all share one
weight matrix.

In [ ]:
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
SPECIAL = ["<sep>"] + [f"<c{i}>" for i in range(MAX_COLS)]


def read_pairs(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def train_tokenizer(train_path="pairs/train_pairs.jsonl", vocab_size=8000, model_prefix="sql_sp"):
    pairs = read_pairs(train_path)
    with open("spm_corpus.txt", "w", encoding="utf-8") as f:
        for p in pairs:  # ONE shared vocabulary for source and target, as in the original Transformer
            f.write(p["src"] + "\n")
            f.write(p["tgt"] + "\n")
    spm.SentencePieceTrainer.train(
        input="spm_corpus.txt",
        model_prefix=model_prefix,
        vocab_size=vocab_size,
        model_type="bpe",
        character_coverage=1.0,
        user_defined_symbols=SPECIAL,  # never split <sep>, <c0>, ...
        pad_id=PAD_ID, unk_id=UNK_ID, bos_id=BOS_ID, eos_id=EOS_ID,
        minloglevel=2,
    )
    return spm.SentencePieceProcessor(model_file=f"{model_prefix}.model")


sp = train_tokenizer(vocab_size=VOCAB_SIZE)
print("vocab size:", sp.get_piece_size())

p = read_pairs("pairs/dev_pairs.jsonl")[0]
print(sp.encode(p["src"], out_type=str))
print(sp.encode(p["tgt"], out_type=str))
print("round-trip ok:", sp.decode(sp.encode(p["tgt"])) == p["tgt"])


### 1.2 (cont.) - `dataset.py` (starter code, unchanged)

In [ ]:
from torch.utils.data import Dataset, DataLoader


class SQLDataset(Dataset):
    """Each item: (source ids, target ids). Target = <s> ... </s>.
    train=True drops over-long pairs.
    train=False keeps EVERY row in file order (needed for evaluation)."""

    def __init__(self, path, sp, train=True, max_src=160, max_tgt=64):
        self.items, skipped = [], 0
        for p in read_pairs(path):
            src = sp.encode(p["src"]) + [EOS_ID]
            tgt = [BOS_ID] + sp.encode(p["tgt"]) + [EOS_ID]
            if train and (len(src) > max_src or len(tgt) > max_tgt):
                skipped += 1
                continue
            self.items.append((src, tgt))
        print(f"{path}: kept {len(self.items)}, skipped {skipped}")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def collate(batch):
    srcs, tgts = zip(*batch)
    S, T = max(map(len, srcs)), max(map(len, tgts))
    src = torch.full((len(batch), S), PAD_ID, dtype=torch.long)
    tgt = torch.full((len(batch), T), PAD_ID, dtype=torch.long)
    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src[i, :len(s)] = torch.tensor(s)
        tgt[i, :len(t)] = torch.tensor(t)
    return src, tgt  # (B, S), (B, T)


def make_loader(path, sp, train, batch_size=64):
    return DataLoader(SQLDataset(path, sp, train=train), batch_size=batch_size, shuffle=train, collate_fn=collate)


BATCH_SIZE = 64 if USE_REAL_DATA else 16
train_dl = make_loader("pairs/train_pairs.jsonl", sp, train=True, batch_size=BATCH_SIZE)
dev_dl = make_loader("pairs/dev_pairs.jsonl", sp, train=False, batch_size=BATCH_SIZE)
test_dl = make_loader("pairs/test_pairs.jsonl", sp, train=False, batch_size=BATCH_SIZE)


**Task 1.3 (part 2) - token-level lengths, on the data actually fed to the model**

In [ ]:
def _token_len_stats(dl):
    src_lens = [len(s) for s, t in dl.dataset.items]
    tgt_lens = [len(t) for s, t in dl.dataset.items]
    return src_lens, tgt_lens

src_lens, tgt_lens = _token_len_stats(train_dl)
print(f"train (kept): mean/max source tokens = {np.mean(src_lens):.1f}/{max(src_lens)}, "
      f"mean/max target tokens = {np.mean(tgt_lens):.1f}/{max(tgt_lens)}")


### 1.2 (cont.) - `embeddings.py` (starter code, unchanged) + `check_starter.py`

In [ ]:
class TokenEmbedding(nn.Module):
    """Section 3.4: embeddings are multiplied by sqrt(d_model).
    One instance is shared by the encoder, the decoder and (optionally) the
    final output projection."""

    def __init__(self, vocab_size, d_model, pad_id=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.scale = math.sqrt(d_model)

    def forward(self, ids):  # (B, L) -> (B, L, d_model)
        return self.emb(ids) * self.scale


class PositionalEncoding(nn.Module):
    """Section 3.5: fixed sinusoidal encodings.
    PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))"""

    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pos = torch.arange(max_len).unsqueeze(1)  # (L, 1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))  # (d/2,)
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, L, d)

    def forward(self, x):  # x: (B, L, d_model)
        return self.dropout(x + self.pe[:, :x.size(1)])


class InputLayer(nn.Module):
    """token ids -> scaled embedding + positional encoding -> dropout"""

    def __init__(self, token_emb, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.tok = token_emb
        self.pos = PositionalEncoding(d_model, max_len, dropout)

    def forward(self, ids):
        return self.pos(self.tok(ids))


# check_starter.py, verbatim behaviour:
src, tgt = next(iter(train_dl))
print("src", tuple(src.shape), "tgt", tuple(tgt.shape))

D_MODEL = 256
_shared_probe = TokenEmbedding(sp.get_piece_size(), D_MODEL, PAD_ID)
_enc_probe = InputLayer(_shared_probe, D_MODEL)
_dec_probe = InputLayer(_shared_probe, D_MODEL)

x = _enc_probe(src)            # (B, S, 256) -> goes into the encoder
y = _dec_probe(tgt[:, :-1])    # (B, T-1, 256) -> goes into the decoder
print("encoder input", tuple(x.shape), "decoder input", tuple(y.shape))


### 1.4 - positional-encoding heat-map

In [ ]:
_pe_probe = PositionalEncoding(d_model=256, max_len=200, dropout=0.0)
pe_grid = _pe_probe.pe[0, :100, :].numpy()  # first 100 positions x 256 dims

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pe_grid.T, aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
ax.set_xlabel("position")
ax.set_ylabel("dimension")
ax.set_title("Sinusoidal positional encoding (first 100 positions x 256 dims)")
fig.colorbar(im, ax=ax, label="value")
fig.tight_layout()
Path("results").mkdir(exist_ok=True)
fig.savefig("results/positional_encoding.png", dpi=150)
plt.show()


**What it shows.** Low-index dimensions oscillate fast (short period) and
higher-index dimensions oscillate progressively slower, so nearby positions
share similar low-frequency components while staying distinguishable in the
high-frequency ones. Since every dimension is a sinusoid, `PE(pos+k)` is a
fixed linear function of `PE(pos)`, which is why the model can attend by
relative position without positions being learned parameters.

## Task 2 - Implement the Transformer

Each component is its own `nn.Module`, matching Section 3 of the paper, with this exact configuration:

| | |
|---|---|
| d_model | 256 |
| heads h | 4 (dk = dv = 64) |
| encoder / decoder layers N | 3 / 3 |
| feed-forward inner size d_ff | 1024 |
| dropout | 0.1 |
| normalisation | post-norm, `LayerNorm(x + Sublayer(x))` |
| weight sharing | encoder embedding = decoder embedding = output projection |


### 2.1 / 2.2 - scaled dot-product and multi-head attention

In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None):
    """Section 3.2.1: Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V

    q: (B, h, Lq, dk)   k: (B, h, Lk, dk)   v: (B, h, Lk, dv)
    mask: broadcastable to (B, h, Lq, Lk), nonzero/True at positions to KEEP.
    Returns (output: (B, h, Lq, dv), attn_weights: (B, h, Lq, Lk)).
    """
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    attn = torch.softmax(scores, dim=-1)
    output = torch.matmul(attn, v)
    return output, attn


class MultiHeadAttention(nn.Module):
    """Section 3.2.2: project Q,K,V with W^Q_i,W^K_i,W^V_i, run h heads in
    parallel, concatenate, project with W^O."""

    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.h = num_heads
        self.d_k = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def _split_heads(self, x):
        B, L, _ = x.shape
        return x.view(B, L, self.h, self.d_k).transpose(1, 2)  # (B, h, L, d_k)

    def _merge_heads(self, x):
        B, h, L, d_k = x.shape
        return x.transpose(1, 2).contiguous().view(B, L, h * d_k)

    def forward(self, query, key, value, mask=None):
        q = self._split_heads(self.w_q(query))
        k = self._split_heads(self.w_k(key))
        v = self._split_heads(self.w_v(value))

        out, attn = scaled_dot_product_attention(q, k, v, mask)
        out = self._merge_heads(out)
        return self.w_o(self.dropout(out)), attn


### 2.3 - position-wise feed-forward, and the post-norm add & norm sublayer

In [ ]:
class PositionwiseFeedForward(nn.Module):
    """Section 3.3: FFN(x) = max(0, xW1 + b1)W2 + b2"""

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(self.relu(self.linear1(x))))


class AddNorm(nn.Module):
    """Post-norm residual connection: LayerNorm(x + Dropout(Sublayer(x)))."""

    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer_out):
        return self.norm(x + self.dropout(sublayer_out))


### 2.4 - masks

In [ ]:
def make_padding_mask(ids, pad_id=0):
    """(B, L) -> (B, 1, 1, L), 1 at real tokens, 0 at <pad>. Broadcasts over
    heads and query positions in scaled_dot_product_attention."""
    return (ids != pad_id).unsqueeze(1).unsqueeze(2)


def make_causal_mask(length, device):
    """(1, 1, L, L) lower-triangular mask: decoder position t may only see
    positions <= t."""
    mask = torch.tril(torch.ones(length, length, device=device, dtype=torch.bool))
    return mask.unsqueeze(0).unsqueeze(0)

### 2.5 / 2.6 - encoder layer and decoder layer

In [ ]:
class EncoderLayer(nn.Module):
    """self-attention -> add & norm -> FFN -> add & norm"""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.add_norm1 = AddNorm(d_model, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)

    def forward(self, x, src_mask):
        attn_out, attn_w = self.self_attn(x, x, x, src_mask)
        x = self.add_norm1(x, attn_out)
        ffn_out = self.ffn(x)
        x = self.add_norm2(x, ffn_out)
        return x, attn_w


class DecoderLayer(nn.Module):
    """masked self-attention -> add & norm -> cross-attention -> add & norm
    -> FFN -> add & norm"""

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.add_norm1 = AddNorm(d_model, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.add_norm3 = AddNorm(d_model, dropout)

    def forward(self, x, enc_out, tgt_mask, cross_mask):
        self_out, self_attn_w = self.self_attn(x, x, x, tgt_mask)
        x = self.add_norm1(x, self_out)
        cross_out, cross_attn_w = self.cross_attn(x, enc_out, enc_out, cross_mask)
        x = self.add_norm2(x, cross_out)
        ffn_out = self.ffn(x)
        x = self.add_norm3(x, ffn_out)
        return x, self_attn_w, cross_attn_w


### 2.7 - full model

In [ ]:
class Encoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

    def forward(self, x, src_mask):
        attn_weights = []
        for layer in self.layers:
            x, attn_w = layer(x, src_mask)
            attn_weights.append(attn_w)
        return x, attn_weights


class Decoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

    def forward(self, x, enc_out, tgt_mask, cross_mask):
        self_attns, cross_attns = [], []
        for layer in self.layers:
            x, self_attn_w, cross_attn_w = layer(x, enc_out, tgt_mask, cross_mask)
            self_attns.append(self_attn_w)
            cross_attns.append(cross_attn_w)
        return x, self_attns, cross_attns


class Transformer(nn.Module):
    """Full encoder-decoder Transformer (Vaswani et al., 2017) on top of the
    starter InputLayer/TokenEmbedding above."""

    def __init__(self, enc_input_layer, dec_input_layer, shared_token_emb,
                 vocab_size, d_model=256, num_heads=4, num_enc_layers=3,
                 num_dec_layers=3, d_ff=1024, dropout=0.1, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.enc_input = enc_input_layer
        self.dec_input = dec_input_layer
        self.encoder = Encoder(num_enc_layers, d_model, num_heads, d_ff, dropout)
        self.decoder = Decoder(num_dec_layers, d_model, num_heads, d_ff, dropout)

        # Output projection shares weights with the token embedding (Section 3.4).
        self.generator = nn.Linear(d_model, vocab_size, bias=False)
        self.generator.weight = shared_token_emb.emb.weight

    def encode(self, src):
        src_mask = make_padding_mask(src, self.pad_id)
        x = self.enc_input(src)
        enc_out, _ = self.encoder(x, src_mask)
        return enc_out, src_mask

    def decode(self, tgt_in, enc_out, src_mask):
        B, T = tgt_in.shape
        pad_mask = make_padding_mask(tgt_in, self.pad_id)
        causal_mask = make_causal_mask(T, tgt_in.device)
        tgt_mask = pad_mask & causal_mask
        x = self.dec_input(tgt_in)
        dec_out, self_attns, cross_attns = self.decoder(x, enc_out, tgt_mask, src_mask)
        return dec_out, self_attns, cross_attns

    def forward(self, src, tgt_in):
        enc_out, src_mask = self.encode(src)
        dec_out, self_attns, cross_attns = self.decode(tgt_in, enc_out, src_mask)
        logits = self.generator(dec_out)
        return logits, cross_attns

    def count_trainable_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


NUM_HEADS = 4
NUM_LAYERS = 3
D_FF = 1024
DROPOUT = 0.1


def build_model(vocab_size, device, d_model=D_MODEL, num_heads=NUM_HEADS,
                 num_layers=NUM_LAYERS, d_ff=D_FF, dropout=DROPOUT):
    shared_emb = TokenEmbedding(vocab_size, d_model, PAD_ID)
    enc_input = InputLayer(shared_emb, d_model, dropout=dropout)
    dec_input = InputLayer(shared_emb, d_model, dropout=dropout)
    model = Transformer(enc_input, dec_input, shared_emb, vocab_size,
                         d_model=d_model, num_heads=num_heads,
                         num_enc_layers=num_layers, num_dec_layers=num_layers,
                         d_ff=d_ff, dropout=dropout, pad_id=PAD_ID)
    return model.to(device)


model = build_model(sp.get_piece_size(), DEVICE)
print(f"Config: d_model={D_MODEL}, heads={NUM_HEADS}, N={NUM_LAYERS}, d_ff={D_FF}")
print(f"Trainable parameters: {model.count_trainable_parameters():,}")

### Correctness checks (assignment Section "Correctness checks")

These only need forward passes on random tensors, so they run immediately -
no training required.

In [ ]:
# 1) Causal mask: changing the LAST token of a decoder input must not change
#    the decoder's outputs at any EARLIER position.
model.eval()
B, T = 2, 6
_vocab = sp.get_piece_size()
dummy_src = torch.randint(4, _vocab, (B, 10)).to(DEVICE)
dummy_tgt = torch.randint(4, _vocab, (B, T)).to(DEVICE)

with torch.no_grad():
    enc_out, src_mask = model.encode(dummy_src)
    dec_out_a, _, _ = model.decode(dummy_tgt, enc_out, src_mask)

    dummy_tgt_changed = dummy_tgt.clone()
    dummy_tgt_changed[:, -1] = (dummy_tgt_changed[:, -1] + 1) % _vocab
    dec_out_b, _, _ = model.decode(dummy_tgt_changed, enc_out, src_mask)

earlier_positions_match = torch.allclose(dec_out_a[:, :-1], dec_out_b[:, :-1], atol=1e-6)
print("Causal mask check (earlier outputs unchanged when last token changes):", earlier_positions_match)
assert earlier_positions_match


In [ ]:
# 2) Padding mask: appending extra <pad> to the source must not change the
#    model's output at the original positions, and attention rows must sum to
#    1 over the unmasked (non-pad) positions.
with torch.no_grad():
    enc_out1, src_mask1 = model.encode(dummy_src)
    dec_out1, _, cross1 = model.decode(dummy_tgt, enc_out1, src_mask1)

    pad_amount = 5
    padded_src = torch.cat([dummy_src, torch.zeros(B, pad_amount, dtype=torch.long, device=DEVICE)], dim=1)
    enc_out2, src_mask2 = model.encode(padded_src)
    dec_out2, _, cross2 = model.decode(dummy_tgt, enc_out2, src_mask2)

output_unchanged = torch.allclose(dec_out1, dec_out2, atol=1e-5)
print("Padding mask check (decoder output unchanged after padding the source):", output_unchanged)
assert output_unchanged

last_layer_attn = cross2[-1]  # (B, h, T, S+pad)
row_sums = last_layer_attn.sum(-1)
print("Cross-attention rows sum to 1 (softmax over unmasked positions):",
      torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5))
attn_on_pad = last_layer_attn[..., -pad_amount:].sum().item()
print(f"Attention mass placed on the padded positions (should be ~0): {attn_on_pad:.2e}")


In [ ]:
# 3) Weight sharing: output projection weight and embedding weight are the
#    SAME tensor object (an `is` check, not just equal values).
same_object = model.generator.weight is model.enc_input.tok.emb.weight
print("Weight sharing check (generator.weight is embedding.weight):", same_object)
assert same_object


## Task 3 - Train

- **3.1 Teacher forcing**: decoder input `tgt[:, :-1]`, prediction target `tgt[:, 1:]`.
- **3.2 Loss**: cross-entropy with label smoothing `eps_ls = 0.1`, ignoring `<pad>`.
- **3.3 Optimiser**: Adam, `beta1=0.9, beta2=0.98, eps=1e-9`.
- **3.4 LR schedule** (paper, Eq. 3), `warmup_steps = 4000`.
- **3.5 Batch size 64, 20 epochs** on a Colab/Kaggle GPU; log train/dev loss
  and LR every epoch; keep the checkpoint with the lowest dev loss.
- **3.6** Dev picks the checkpoint; test is touched once, at the end.

In [ ]:
WARMUP_STEPS = 4000
LABEL_SMOOTHING = 0.1


def noam_lr(step, d_model=D_MODEL, warmup=WARMUP_STEPS):
    """Eq. 3: lrate = d_model^-0.5 * min(step^-0.5, step*warmup^-1.5)."""
    step = max(step, 1)
    return d_model ** -0.5 * min(step ** -0.5, step * warmup ** -1.5)


steps = np.arange(1, 20001)
lrs = [noam_lr(int(s)) for s in steps]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(steps, lrs)
ax.axvline(WARMUP_STEPS, color="gray", linestyle="--", label=f"warmup = {WARMUP_STEPS}")
ax.set_xlabel("training step")
ax.set_ylabel("learning rate")
ax.set_title("Noam learning-rate schedule (Eq. 3)")
ax.legend()
fig.tight_layout()
fig.savefig("results/lr_schedule.png", dpi=150)
plt.show()

rising = all(lrs[i] < lrs[i + 1] for i in range(0, WARMUP_STEPS - 2, 500))
decaying = all(lrs[i] > lrs[i + 1] for i in range(WARMUP_STEPS + 500, len(lrs) - 500, 500))
print("Rises for the first", WARMUP_STEPS, "steps:", rising)
print("Decays afterwards:", decaying)


In [ ]:
def run_epoch(model, loader, optimizer, scheduler, criterion, device, train):
    model.train(train)
    total_loss, total_tokens = 0.0, 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]

        with torch.set_grad_enabled(train):
            logits, _ = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

        n_tokens = (tgt_out != PAD_ID).sum().item()
        if train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

        total_loss += loss.item() * n_tokens
        total_tokens += n_tokens
    return total_loss / max(total_tokens, 1)


# NOTE: NUM_EPOCHS is small by default so a top-to-bottom smoke test finishes
# quickly on CPU too. Set it to 20 (the assignment's spec) for the graded run
# on a Kaggle/Colab GPU with the real WikiSQL data.
NUM_EPOCHS = 20 if USE_REAL_DATA else 8

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=noam_lr)

CKPT_PATH = "results/best.pt"
best_dev_loss = float("inf")
train_log = []
for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss = run_epoch(model, train_dl, optimizer, scheduler, criterion, DEVICE, train=True)
    dev_loss = run_epoch(model, dev_dl, optimizer, scheduler, criterion, DEVICE, train=False)
    lr = optimizer.param_groups[0]["lr"]
    elapsed = time.time() - t0
    print(f"epoch {epoch:2d} | train_loss {train_loss:.4f} | dev_loss {dev_loss:.4f} | lr {lr:.2e} | {elapsed:.1f}s")
    train_log.append({"epoch": epoch, "train_loss": train_loss, "dev_loss": dev_loss, "lr": lr, "seconds": elapsed})
    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        torch.save({"model": model.state_dict(), "epoch": epoch, "dev_loss": dev_loss,
                    "vocab_size": sp.get_piece_size()}, CKPT_PATH)

with open("results/train_log.json", "w") as f:
    json.dump(train_log, f, indent=2)
print("\nBest dev loss:", best_dev_loss, "-> saved to", CKPT_PATH)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([h["epoch"] for h in train_log], [h["train_loss"] for h in train_log], label="train loss")
ax.plot([h["epoch"] for h in train_log], [h["dev_loss"] for h in train_log], label="dev loss")
ax.set_xlabel("epoch")
ax.set_ylabel("label-smoothed cross-entropy")
ax.set_title("Training and dev loss per epoch")
ax.legend()
fig.tight_layout()
fig.savefig("results/loss_curve.png", dpi=150)
plt.show()

model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE)["model"])
model.eval()
print("Loaded best checkpoint (epoch", torch.load(CKPT_PATH, map_location=DEVICE)["epoch"], ")")


## Task 4 - Decode and convert to SQL

- **4.1** Greedy decoding and beam search (beam size 4), stopping at `</s>`
  or after 64 tokens.
- **4.2** Detokenise with `sp.decode`, parse back into
  `{"sel": int, "agg": int, "conds": [[col, op, value], ...]}`.
- **4.3** One JSON line per example, same order as the input file; unparsable
  lines become `{"error": "parse"}` - never skipped.
- **4.4** Turn the parsed query back into SQL with real column names.

In [ ]:
@torch.no_grad()
def greedy_decode(model, src, max_len=64, device="cpu"):
    """src: (B, S). Returns (token ids (B, <=max_len+1), last-call cross-attn)."""
    model.eval()
    src = src.to(device)
    enc_out, src_mask = model.encode(src)
    B = src.size(0)
    ys = torch.full((B, 1), BOS_ID, dtype=torch.long, device=device)
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    cross_attns = None

    for _ in range(max_len):
        dec_out, _, cross_attns = model.decode(ys, enc_out, src_mask)
        logits = model.generator(dec_out[:, -1])
        next_tok = logits.argmax(-1, keepdim=True)
        next_tok = torch.where(finished.unsqueeze(-1), torch.full_like(next_tok, EOS_ID), next_tok)
        ys = torch.cat([ys, next_tok], dim=1)
        finished = finished | (next_tok.squeeze(-1) == EOS_ID)
        if finished.all():
            break
    return ys, cross_attns


@torch.no_grad()
def beam_search_decode(model, src, beam_size=4, max_len=64, device="cpu"):
    """Beam search for a single example. src: (1, S)."""
    model.eval()
    src = src.to(device)
    enc_out, src_mask = model.encode(src)

    beams = [(torch.tensor([[BOS_ID]], device=device), 0.0, False)]
    for _ in range(max_len):
        candidates = []
        for ys, score, done in beams:
            if done:
                candidates.append((ys, score, done))
                continue
            dec_out, _, _ = model.decode(ys, enc_out, src_mask)
            logits = model.generator(dec_out[:, -1])
            log_probs = torch.log_softmax(logits, dim=-1).squeeze(0)
            topk_logp, topk_idx = log_probs.topk(min(beam_size, log_probs.size(-1)))
            for lp, idx in zip(topk_logp, topk_idx):
                new_ys = torch.cat([ys, idx.view(1, 1)], dim=1)
                candidates.append((new_ys, score + lp.item(), idx.item() == EOS_ID))
        candidates.sort(key=lambda c: c[1] / c[0].size(1), reverse=True)
        beams = candidates[:beam_size]
        if all(done for _, _, done in beams):
            break

    best_ys = beams[0][0]
    _, _, cross_attns = model.decode(best_ys, enc_out, src_mask)
    return best_ys, cross_attns


def parse_prediction(text):
    """'select count <c3> where <c1> = kim manners' -> {'sel','agg','conds'} or None."""
    try:
        tokens = text.strip().split()
        assert tokens and tokens[0] == "select"
        i = 1
        agg = 0
        agg_names = [a.lower() for a in AGG_OPS if a]
        if tokens[i] in agg_names:
            agg = [a.lower() for a in AGG_OPS].index(tokens[i])
            i += 1

        sel_tok = tokens[i]
        assert sel_tok.startswith("<c") and sel_tok.endswith(">")
        sel = int(sel_tok[2:-1])
        i += 1

        conds = []
        while i < len(tokens):
            assert tokens[i] in ("where", "and")
            i += 1
            col_tok = tokens[i]
            assert col_tok.startswith("<c") and col_tok.endswith(">")
            col = int(col_tok[2:-1])
            i += 1
            op = tokens[i]
            assert op in COND_OPS
            i += 1
            val_tokens = []
            while i < len(tokens) and tokens[i] != "and":
                val_tokens.append(tokens[i])
                i += 1
            value = " ".join(val_tokens)
            assert value != ""
            conds.append([col, COND_OPS.index(op), value])
        return {"sel": sel, "agg": agg, "conds": conds}
    except (AssertionError, IndexError, ValueError):
        return None


def readable_sql(parsed, header):
    """Turn a parsed query back into SQL using the real column names."""
    agg = AGG_OPS[parsed["agg"]]
    col = header[parsed["sel"]]
    sel_expr = f"{agg}({col})" if agg else col
    sql = f"SELECT {sel_expr} FROM table"
    if parsed["conds"]:
        clauses = [f"{header[c]} {COND_OPS[op]} '{val}'" for c, op, val in parsed["conds"]]
        sql += " WHERE " + " AND ".join(clauses)
    return sql


def _strip_special(ids):
    if EOS_ID in ids:
        ids = ids[:ids.index(EOS_ID)]
    if ids and ids[0] == BOS_ID:
        ids = ids[1:]
    return ids


def decode_example(model, sp, src_row, method="greedy", beam_size=4, max_len=64, device="cpu"):
    if method == "greedy":
        ys, cross_attns = greedy_decode(model, src_row, max_len=max_len, device=device)
    else:
        ys, cross_attns = beam_search_decode(model, src_row, beam_size=beam_size, max_len=max_len, device=device)
    ids = _strip_special(ys[0].tolist())
    text = sp.decode(ids)
    parsed = parse_prediction(text)
    prediction = {"error": "parse"} if parsed is None else {"query": parsed}
    return prediction, text, cross_attns


def decode_split(model, loader, sp, device, method="greedy", beam_size=4, max_len=64):
    """One prediction dict per example, IN ORDER (loader must have train=False).
    Beam search in particular can take a while over a full split, so this
    prints periodic progress rather than running silently."""
    predictions = []
    n_total = len(loader.dataset)
    report_every = max(1, n_total // 10)
    t0 = time.time()
    for src, _ in loader:
        for i in range(src.size(0)):
            pred, _, _ = decode_example(model, sp, src[i:i + 1], method, beam_size, max_len, device)
            predictions.append(pred)
            if len(predictions) % report_every == 0 or len(predictions) == n_total:
                print(f"  decoded {len(predictions)}/{n_total} ({method}) - {time.time() - t0:.0f}s elapsed")
    return predictions


def write_predictions(path, predictions):
    with open(path, "w", encoding="utf-8") as f:
        for p in predictions:
            f.write(json.dumps(p) + "\n")


In [ ]:
dev_examples, dev_tables = load_split("dev")

example = dev_examples[0]
table = dev_tables[example["table_id"]]
src_ids, _ = dev_dl.dataset[0]
src_row = torch.tensor([src_ids])

for method in ["greedy", "beam"]:
    pred, text, _ = decode_example(model, sp, src_row, method=method, beam_size=4, max_len=64, device=DEVICE)
    print(f"--- {method} ---")
    print("question:", example["question"])
    print("generated (tokenised):", text)
    print("parsed:", pred)
    if "query" in pred:
        print("readable SQL:", readable_sql(pred["query"], table["header"]))
    print()


### Parser round-trip sanity check

Feeding the **gold** target string through the same parser used on model
output should always recover the original `(sel, agg, conds)` - if it
didn't, a correct model could still be scored as wrong.

In [ ]:
ok = 0
for ex in dev_examples:
    tgt_str = encode_target(ex["sql"])
    parsed = parse_prediction(tgt_str)
    gold_conds = {(c[0], c[1], str(c[2]).lower()) for c in ex["sql"]["conds"]}
    pred_conds = {(c[0], c[1], str(c[2]).lower()) for c in parsed["conds"]} if parsed else set()
    if parsed and parsed["sel"] == ex["sql"]["sel"] and parsed["agg"] == ex["sql"]["agg"] and pred_conds == gold_conds:
        ok += 1
print(f"Gold parser round-trip: {ok}/{len(dev_examples)} = {100*ok/len(dev_examples):.1f}%")
print("The assignment's actual correctness check plugs this into WikiSQL's own")
print("evaluate.py + dev.db for a true EXECUTION-accuracy round trip (must be > 99%) - see Task 5.")


## Task 5 - Evaluate and analyse

- **5.1** Run the **official** WikiSQL evaluator (never a reimplementation)
  on dev (greedy and beam) and once on test.
- **5.2** Component accuracy: % correct `sel`, % correct `agg`, % correct
  WHERE clause (same set of conditions, order ignored).
- **5.3** Cross-attention map for one dev example.

In [ ]:
def parse_failure_rate(predictions):
    n_fail = sum(1 for p in predictions if "error" in p)
    return 100 * n_fail / len(predictions) if predictions else 0.0


def component_accuracy(predictions, gold_examples):
    n = len(gold_examples)
    sel_correct = agg_correct = where_correct = 0
    for pred, gold in zip(predictions, gold_examples):
        if "error" in pred:
            continue
        p, g = pred["query"], gold["sql"]
        if p["sel"] == g["sel"]:
            sel_correct += 1
        if p["agg"] == g["agg"]:
            agg_correct += 1
        pred_conds = {(c[0], c[1], str(c[2]).lower()) for c in p["conds"]}
        gold_conds = {(c[0], c[1], str(c[2]).lower()) for c in g["conds"]}
        if pred_conds == gold_conds:
            where_correct += 1
    return {"sel_acc": 100 * sel_correct / n if n else 0.0,
            "agg_acc": 100 * agg_correct / n if n else 0.0,
            "where_acc": 100 * where_correct / n if n else 0.0}


def run_official_evaluator(wikisql_dir, pred_path, split):
    """Shells out to WikiSQL's own evaluate.py ("do not write your own").
    Needs `wikisql_dir` to be a clone of https://github.com/salesforce/WikiSQL
    with data.tar.bz2 already extracted (Task 1.1)."""
    cmd = [sys.executable, "evaluate.py", f"data/{split}.jsonl", f"data/{split}.db",
           str(Path(pred_path).resolve())]
    result = subprocess.run(cmd, cwd=wikisql_dir, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr, file=sys.stderr)
    return result.stdout


dev_predictions_greedy = decode_split(model, dev_dl, sp, DEVICE, method="greedy", max_len=64)
write_predictions("results/dev_greedy.jsonl", dev_predictions_greedy)

metrics_greedy = component_accuracy(dev_predictions_greedy, dev_examples)
metrics_greedy["parse_failure_rate"] = parse_failure_rate(dev_predictions_greedy)
print("Dev component accuracy (greedy):")
for k, v in metrics_greedy.items():
    print(f"  {k}: {v:.1f}%")


### 5.1 - official evaluator on dev (greedy and beam)

Decodes dev with beam search too, then calls WikiSQL's own `evaluate.py` on both prediction files - never a reimplementation of the metric. Needs the `WikiSQL` folder from the Task 1.1 clone (code + data), not just the data.

In [ ]:
dev_predictions_beam = decode_split(model, dev_dl, sp, DEVICE, method="beam", beam_size=4, max_len=64)
write_predictions("results/dev_beam.jsonl", dev_predictions_beam)

metrics_beam = component_accuracy(dev_predictions_beam, dev_examples)
metrics_beam["parse_failure_rate"] = parse_failure_rate(dev_predictions_beam)
print("Dev component accuracy (beam):")
for k, v in metrics_beam.items():
    print(f"  {k}: {v:.1f}%")

WIKISQL_REPO_DIR = Path("WikiSQL")
if (WIKISQL_REPO_DIR / "evaluate.py").exists():
    print("\n=== Official evaluator: dev, greedy ===")
    run_official_evaluator(WIKISQL_REPO_DIR, "results/dev_greedy.jsonl", "dev")
    print("=== Official evaluator: dev, beam ===")
    run_official_evaluator(WIKISQL_REPO_DIR, "results/dev_beam.jsonl", "dev")
else:
    print(f"No WikiSQL code repo found at {WIKISQL_REPO_DIR.resolve()}.")
    print("Re-run the Task 1.1 cell (git clone) to get evaluate.py, then re-run this cell.")


### Correctness check - gold round-trip via the official evaluator

Gold dev targets, parsed and run through WikiSQL's own `evaluate.py` - execution accuracy here must be above 99%, or the parser/format is broken independently of the model.

In [ ]:
gold_predictions = []
for ex in dev_examples:
    parsed = parse_prediction(encode_target(ex["sql"]))
    gold_predictions.append({"error": "parse"} if parsed is None else {"query": parsed})
write_predictions("results/dev_gold_roundtrip.jsonl", gold_predictions)

if (WIKISQL_REPO_DIR / "evaluate.py").exists():
    print("=== Official evaluator: dev, GOLD round-trip (must be > 99% execution) ===")
    run_official_evaluator(WIKISQL_REPO_DIR, "results/dev_gold_roundtrip.jsonl", "dev")
else:
    print("No WikiSQL code repo found - skipping the gold round-trip check.")


### 5.1 (cont.) - test, once, with the final decoding choice

Set `FINAL_METHOD` from whichever of greedy/beam scored higher on dev above, then decode and evaluate test exactly once.

In [ ]:
FINAL_METHOD = "beam"  # set to "greedy" instead if that scored higher on dev above

test_examples, test_tables = load_split("test")
test_predictions = decode_split(model, test_dl, sp, DEVICE, method=FINAL_METHOD, beam_size=4, max_len=64)
write_predictions("results/test_final.jsonl", test_predictions)

test_metrics = component_accuracy(test_predictions, test_examples)
test_metrics["parse_failure_rate"] = parse_failure_rate(test_predictions)
print(f"Test component accuracy ({FINAL_METHOD}):")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.1f}%")

if (WIKISQL_REPO_DIR / "evaluate.py").exists():
    print(f"\n=== Official evaluator: test, {FINAL_METHOD} ===")
    run_official_evaluator(WIKISQL_REPO_DIR, "results/test_final.jsonl", "test")


### 5.3 - cross-attention map

In [ ]:
def plot_cross_attention(cross_attn_last_layer, src_tokens, gen_tokens, out_path):
    """cross_attn_last_layer: (1, h, T, S), averaged over heads -> (T, S)."""
    attn = cross_attn_last_layer[0].mean(0).detach().cpu().numpy()
    fig, ax = plt.subplots(figsize=(max(6, len(src_tokens) * 0.5), max(4, len(gen_tokens) * 0.4)))
    im = ax.imshow(attn, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(src_tokens))); ax.set_xticklabels(src_tokens, rotation=90, fontsize=8)
    ax.set_yticks(range(len(gen_tokens))); ax.set_yticklabels(gen_tokens, fontsize=8)
    ax.set_xlabel("source tokens"); ax.set_ylabel("generated tokens")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    return attn


pred, text, cross_attns = decode_example(model, sp, src_row, method="greedy", max_len=64, device=DEVICE)
last_layer_cross_attn = cross_attns[-1]  # (1, h, T, S)

src_tokens = sp.encode(all_pairs["dev"][0]["src"], out_type=str) + ["</s>"]
gen_tokens = sp.encode(text, out_type=str)

T_actual, S_actual = last_layer_cross_attn.shape[-2], last_layer_cross_attn.shape[-1]
gen_labels = (gen_tokens + ["-"] * T_actual)[:T_actual]
src_labels = (src_tokens + ["-"] * S_actual)[:S_actual]

plot_cross_attention(last_layer_cross_attn, src_labels, gen_labels, "results/cross_attention.png")
print("question:", example["question"])
print("generated:", text)


### Qualitative samples

Ten dev examples with question, gold SQL, generated SQL, and (for wrong
ones) the failure type. Written to `results/samples.md`.

In [ ]:
# Curate 5 correct and 5 wrong predictions (searching dev in order), rather
# than just the first 10 rows - a fixed-position slice can easily land on
# all-wrong or all-right by chance and isn\'t a fair qualitative sample.
correct_rows, wrong_rows = [], []
SEARCH_CAP = min(len(dev_examples), 2000)

for i in range(SEARCH_CAP):
    if len(correct_rows) >= 5 and len(wrong_rows) >= 5:
        break
    ex = dev_examples[i]
    tab = dev_tables[ex["table_id"]]
    src_ids_i, _ = dev_dl.dataset[i]
    pred, text, _ = decode_example(model, sp, torch.tensor([src_ids_i]), method="greedy", max_len=64, device=DEVICE)
    gold_sql = readable_sql(ex["sql"], tab["header"])
    if "query" in pred:
        pred_sql = readable_sql(pred["query"], tab["header"])
        correct = (pred["query"]["sel"] == ex["sql"]["sel"] and pred["query"]["agg"] == ex["sql"]["agg"] and
                   {(c[0], c[1], str(c[2]).lower()) for c in pred["query"]["conds"]} ==
                   {(c[0], c[1], str(c[2]).lower()) for c in ex["sql"]["conds"]})
    else:
        pred_sql, correct = "(parse failure)", False

    row = (ex["question"], gold_sql, pred_sql, correct)
    if correct and len(correct_rows) < 5:
        correct_rows.append(row)
    elif not correct and len(wrong_rows) < 5:
        wrong_rows.append(row)

rows = correct_rows + wrong_rows

with open("results/samples.md", "w", encoding="utf-8") as f:
    f.write("| # | Question | Gold SQL | Predicted SQL | Correct |\n|---|---|---|---|---|\n")
    for i, (q, g, p, c) in enumerate(rows, 1):
        f.write(f"| {i} | {q} | `{g}` | `{p}` | {'yes' if c else 'no'} |\n")

for i, (q, g, p, c) in enumerate(rows, 1):
    print(f"{i}. {'OK  ' if c else 'FAIL'} | Q: {q}\n     gold: {g}\n     pred: {p}\n")

print(f"Found {len(correct_rows)} correct and {len(wrong_rows)} wrong "
      f"in the first {SEARCH_CAP} dev examples.")

## Package results

Zips the checkpoint, plots, prediction files and `samples.md` into
`results_bundle.zip`, available from this notebook's **Output** tab on
Kaggle.

In [ ]:
import shutil

# The tokenizer model lives outside results/ (it's written before results/
# exists) - copy it in so the frontend/backend has everything from one zip.
for fname in ["sql_sp.model", "sql_sp.vocab"]:
    if Path(fname).exists():
        shutil.copy(fname, f"results/{fname}")

zip_path = shutil.make_archive("results_bundle", "zip", "results")
print("Zipped results/ ->", zip_path)

import os
print(f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")
